In [ ]:
# Parameters
parser = argparse.ArgumentParser(description = 'Parse parameters for multilocus GWAS')
parser.add_argument('-p', '--phenotype', help = 'Path to phenotype file. CSV file with accession ID in forst column, phenptype values in second column. Filename is used as phenotype name.', required = True)
parser.add_argument('-g', '--genotype', help = 'Path to the SNP matrix in HDF5 format.', required=True)
parser.add_argument('-k', '--kinship', help = 'Path to the kinship matrix in HDF5 format.', required = True)
parser.add_argument('-m', '--maf', help = 'Specify the minor allele frequecny cut-off. Default is set to 0.05', default = 0.05)
parser.add_argument('-o', '--outDir', help = 'Specify the output directory. All results will be saved in this directory.', required = True)
args = parser.parse_args()

In [1]:
!pip install limix

Defaulting to user installation because normal site-packages is not writeable
  Using cached limix-3.0.4-py3-none-any.whl
ERROR: Could not install packages due to an OSError: [Errno 2] No such file or directory: '/users/elizaveta.grigoreva/.local/lib/python3.9/site-packages/packaging-25.0.dist-info/METADATA'



In [2]:
from limix.qtl import scan
import pandas as pd
import numpy as np

ModuleNotFoundError: No module named 'limix'

In [ ]:
import numpy as np

In [9]:
!pip install limix

Defaulting to user installation because normal site-packages is not writeable
  Using cached limix-3.0.4-py3-none-any.whl
ERROR: Could not install packages due to an OSError: [Errno 2] No such file or directory: '/users/elizaveta.grigoreva/.local/lib/python3.9/site-packages/packaging-25.0.dist-info/METADATA'



In [10]:
## Step 1. Phenotype load 
import pandas as pd

pheno_path = "/groups/nordborg/projects/ddm1_resilencing/03_processing/006_gwas/all_plates_96w/limix_gwas/phenotype.txt"  # Phenotype load 
pheno = pd.read_csv(pheno_path, index_col=0, sep="\t", header=None)

print(pheno.head())
print("Shape:", pheno.shape)

              1         2
0                        
10010  0.024980 -0.324621
1002  -0.049857 -0.024401
108   -0.033401  1.064215
14319 -0.040161  0.492114
15560 -0.016326  0.265559
Shape: (266, 2)


In [17]:
import h5py
import numpy as np

geno_file = '/groups/nordborg/projects/ddm1_resilencing/03_processing/006_gwas/all_plates_96w/limix_gwas/1163g.179kB.prior15.gauss4.ts99.5.BIALLELIC.filtered.vcf.hdf5'

with h5py.File(geno_file, 'r') as f:
    samples = f['samples'][:]
    samples = np.array([s.decode() if isinstance(s, bytes) else s for s in samples])
    
    GT = f['calldata/GT'][:]  # (variants, samples, ploidy=2)
    G = GT.sum(axis=2)  # Convert to 0/1/2 genotype dosage

# Make sure pheno index is str
pheno.index = pheno.index.astype(str)

# Find intersection
common_accessions = [acn for acn in pheno.index if acn in samples]
if not common_accessions:
    raise ValueError("No matching accessions between phenotype and genotype!")

# Get indices
acn_indices = [np.where(samples == acn)[0][0] for acn in common_accessions]
G = G[:, acn_indices]
samples_matched = samples[acn_indices]
pheno = pheno.loc[samples_matched]


In [18]:
G.shape

(7051412, 266)

In [19]:
## Filtering by MAF
# Count alternate alleles (AC1) and reference alleles (AC0)
AC1 = G.sum(axis=1)
AC0 = G.shape[1] * 2 - AC1  # diploid: 2 alleles per sample

# Stack for MAF calculation
AC = np.vstack((AC0, AC1))
MAC = np.min(AC, axis=0)  # Minor allele count
MAF = MAC / (G.shape[1] * 2)  # Minor allele frequency

# Filter SNPs with MAF >= 5%
#SNP_indices = np.where(MAF >= 0.00)[0]
#G = G[SNP_indices, :]
G = G.T  # Now shape: (samples, SNPs)
G.shape

(266, 4014812)

In [9]:
## Kinship and check 
import numpy as np
import pandas as pd

# 1. Load kinship matrix
K = np.loadtxt('/groups/nordborg/projects/ddm1_resilencing/03_processing/006_gwas/all_plates_96w/limix_gwas/kinship.rel')

# 2. Load IDs file - now skipping the header row
ids = pd.read_csv('/groups/nordborg/projects/ddm1_resilencing/03_processing/006_gwas/all_plates_96w/limix_gwas/kinship.rel.id',
                  delimiter='\s+',
                  header=0,  # This tells pandas to use the first row as header
                  names=['IID'])  # This names the column

# Convert IDs to strings
ids['IID'] = ids['IID'].astype(str)
pheno_index = pheno.index.astype(str)

# Create index mapping (now without the "IID" header issue)
id_to_index = dict(zip(ids['IID'], range(len(ids))))

# Get indices
acn_indices = [id_to_index[acn] for acn in pheno_index]

# Subset the kinship matrix
K_sub = K[acn_indices, :][:, acn_indices]

print("Original matrix shape:", K.shape)
print("Subsetted matrix shape:", K_sub.shape)

Original matrix shape: (266, 266)
Subsetted matrix shape: (266, 266)


In [21]:
import numpy as np

# G: (samples, SNPs)
# pheno: (samples, 2)
# K_sub: (samples, samples)

Y = pheno.values  # shape: (samples, 2)
X = np.ones((Y.shape[0], 1))  # Intercept only

print("Genotype shape:", G.shape)
print("Phenotype shape:", Y.shape)
print("Kinship shape:", K_sub.shape)

Genotype shape: (266, 4014812)
Phenotype shape: (266, 2)
Kinship shape: (266, 266)


In [1]:
import limix
print(limix.__version__)

ModuleNotFoundError: No module named 'limix'

In [32]:
from limix.qtl import scan
help(scan)

Help on function scan in module limix.qtl._scan:

scan(G, Y, lik='normal', K=None, M=None, idx=None, A=None, A0=None, A1=None, verbose=True)
    Multi-trait association and interaction testing via linear mixed models.
    
    Let n, c, and p be the number of samples, covariates, and traits, respectively.
    The outcome variable Y is a n×p matrix distributed according to ::
    
        vec(Y) ~ N((A ⊗ M) vec(𝚨), K₀ = C₀ ⊗ K + C₁ ⊗ I) under H₀.
    
    A and M are design matrices of dimensions p×p and n×c provided by the user,
    where X is the usual matrix of covariates commonly used in single-trait models.
    𝚨 is a c×p matrix of fixed-effect sizes per trait.
    C₀ and C₁ are both symmetric matrices of dimensions p×p, for which C₁ is
    guaranteed by our implementation to be of full rank.
    The parameters of the H₀ model are the matrices 𝚨, C₀, and C₁.
    
    The additional models H₁ and H₂ are define as ::
    
        vec(Y) ~ N((A ⊗ M) vec(𝚨) + (A₀ ⊗ Gᵢ) vec(𝚩₀), s⋅K₀)
 

In [54]:
G = np.asarray(G)
Y = np.asarray(Y)
K = np.asarray(K_sub)  # or whatever your kinship variable is called

In [59]:
G.shape
Y.shape
K.shape

(266, 266)

In [ ]:
!pip install packaging
!pip install limix
!pip install --force-reinstall --no-deps packaging==25.0


: 

In [ ]:
import packaging
print(packaging.__version__)

23.1


In [ ]:

# G: (samples, SNPs)
# Y: (samples, traits)
# K_sub: (samples, samples)
# X: (samples, covariates)  # intercept

from limix.qtl import scan

result = scan(
    G=G,
    Y=Y,
    lik='normal',
    K=K,
    verbose=True
)

: 

In [5]:
## Kinship and check 
import numpy as np
import pandas as pd

# 1. Load kinship matrix
K = np.loadtxt('/groups/nordborg/projects/ddm1_resilencing/03_processing/006_gwas/all_plates_96w/limix_gwas/kinship.rel')

# 2. Load IDs file - now skipping the header row
ids = pd.read_csv('/groups/nordborg/projects/ddm1_resilencing/03_processing/006_gwas/all_plates_96w/limix_gwas/kinship.rel.id',
                  delimiter='\s+',
                  header=0,  # This tells pandas to use the first row as header
                  names=['IID'])  # This names the column

# Convert IDs to strings
ids['IID'] = ids['IID'].astype(str)
pheno_index = pheno.index.astype(str)

# Create index mapping (now without the "IID" header issue)
id_to_index = dict(zip(ids['IID'], range(len(ids))))

# Get indices
acn_indices = [id_to_index[acn] for acn in pheno_index]

# Subset the kinship matrix
K_sub = K[acn_indices, :][:, acn_indices]

print("Original matrix shape:", K.shape)
print("Subsetted matrix shape:", K_sub.shape)

Original matrix shape: (266, 266)
Subsetted matrix shape: (266, 266)


In [9]:
import h5py 

In [6]:
import h5py
import numpy as np
import pandas as pd

# 1. First check phenotype file
print("Checking phenotype file...")
pheno = pd.read_csv('/groups/nordborg/projects/ddm1_resilencing/03_processing/006_gwas/all_plates_96w/limix_gwas/phenotype.txt', index_col=0)
print("Phenotype data shape:", pheno.shape)
print("First few rows of phenotype data:")
print(pheno.head())
print("\nPhenotype index example:", pheno.index[:5])

# 2. Check HDF5 file structure
print("\nChecking HDF5 file structure...")
with h5py.File('/groups/nordborg/projects/ddm1_resilencing/03_processing/006_gwas/all_plates_96w/limix_gwas/1163g.179kB.prior15.gauss4.ts99.5.BIALLELIC.filtered.vcf.hdf5', 'r') as f:
    print("Available keys:", list(f.keys()))
    # Print structure
    def print_structure(name, obj):
        print(f"Found object: {name}, Type: {type(obj)}")
        if isinstance(obj, h5py.Dataset):
            print(f"    Shape: {obj.shape}")
            print(f"    dtype: {obj.dtype}")
    
    f.visititems(print_structure)

# 3. Check kinship files
print("\nChecking kinship files...")
# Read kinship IDs
ids = pd.read_csv('/groups/nordborg/projects/ddm1_resilencing/03_processing/006_gwas/all_plates_96w/limix_gwas/kinship.rel.id',
                  delimiter='\s+',
                  header=None)
print("Kinship IDs shape:", ids.shape)
print("First few kinship IDs:")
print(ids.head())

# Try to read a small part of the kinship matrix
K = np.loadtxt('/groups/nordborg/projects/ddm1_resilencing/03_processing/006_gwas/all_plates_96w/limix_gwas/kinship.rel')
print("\nKinship matrix shape:", K.shape)

Checking phenotype file...
Phenotype data shape: (265, 0)
First few rows of phenotype data:
Empty DataFrame
Columns: []
Index: [1002	-0.0498567588390866	-0.0244012147190962, 108	-0.0334012310597945	1.06421505652304, 14319	-0.0401607745727262	0.492114242290163, 15560	-0.0163255979759414	0.265559225659471, 15592	-0.087685162558938	0.164224731244577]

Phenotype index example: Index(['1002\t-0.0498567588390866\t-0.0244012147190962',
       '108\t-0.0334012310597945\t1.06421505652304',
       '14319\t-0.0401607745727262\t0.492114242290163',
       '15560\t-0.0163255979759414\t0.265559225659471',
       '15592\t-0.087685162558938\t0.164224731244577'],
      dtype='object', name='10010\t0.0249804652996104\t-0.324620943956854')

Checking HDF5 file structure...
Available keys: ['calldata', 'samples', 'variants']
Found object: calldata, Type: <class 'h5py._hl.group.Group'>
Found object: calldata/AD, Type: <class 'h5py._hl.dataset.Dataset'>
    Shape: (7051412, 266, 4)
    dtype: int16
Found obje

In [7]:
pheno = pd.read_csv(
    '/groups/nordborg/projects/ddm1_resilencing/03_processing/006_gwas/all_plates_96w/limix_gwas/phenotype.txt',
    sep='\t',  # Specify tab delimiter!
    index_col=0,
header=None)
print(pheno.shape)
print(pheno.head())

(266, 2)
              1         2
0                        
10010  0.024980 -0.324621
1002  -0.049857 -0.024401
108   -0.033401  1.064215
14319 -0.040161  0.492114
15560 -0.016326  0.265559


In [8]:
##Import genotype file
import h5py
import numpy as np

geno_file = '/groups/nordborg/projects/ddm1_resilencing/03_processing/006_gwas/all_plates_96w/limix_gwas/1163g.179kB.prior15.gauss4.ts99.5.BIALLELIC.filtered.vcf.hdf5'
with h5py.File(geno_file, 'r') as f:
    samples = f['samples'][:].astype(str)
    GT = f['calldata/GT'][:]  # shape: (variants, samples, 2)
    # Convert GT to 0/1/2 (sum of alleles)
    G = GT.sum(axis=2)  # shape: (variants, samples)

In [9]:
G.shape

(7051412, 266)

In [10]:
# Make sure pheno index and samples are both strings
pheno.index = pheno.index.astype(str)
samples = np.array([s.decode() if isinstance(s, bytes) else s for s in samples])

# Find indices of phenotype accessions in genotype samples
acn_indices = [np.where(samples == acn)[0][0] for acn in pheno.index if acn in samples]
G = G[:, acn_indices]
samples_matched = samples[acn_indices]
pheno = pheno.loc[samples_matched]

In [15]:
G.shape

(7051412, 266)

In [16]:
pheno

,1,2
0,,
10010,0.024980,-0.324621
1002,-0.049857,-0.024401
108,-0.033401,1.064215
14319,-0.040161,0.492114
15560,-0.016326,0.265559
...,...,...
9927,0.015342,-0.406540
9933,0.072331,0.145948
9938,-0.054522,0.056299


In [11]:
## Filtering by MAF
AC1 = G.sum(axis=1)
AC0 = G.shape[1] * 2 - AC1  # diploid: 2 alleles per sample
AC = np.vstack((AC0, AC1))
MAC = np.min(AC, axis=0)
MAF = MAC / (G.shape[1] * 2)
SNP_indices = np.where(MAF >= 0.05)[0]
G = G[SNP_indices, :]

In [18]:
## Transpose
G = G.T  # Now shape: (samples, SNPs)
G.shape

(266, 943557)

In [12]:
K_sub

array([[ 1.21806e+00, -1.92633e-02, -2.79335e-02, ..., -2.86010e-02,
         9.76501e-04, -6.48333e-03],
       [-1.92633e-02,  9.89894e-01, -7.38143e-03, ..., -9.28916e-03,
        -4.21479e-03, -9.27230e-03],
       [-2.79335e-02, -7.38143e-03,  9.11266e-01, ...,  3.70257e-02,
        -8.81522e-03, -6.73866e-03],
       ...,
       [-2.86010e-02, -9.28916e-03,  3.70257e-02, ...,  7.68257e-01,
        -1.13460e-02, -9.31693e-04],
       [ 9.76501e-04, -4.21479e-03, -8.81522e-03, ..., -1.13460e-02,
         9.97550e-01,  9.34093e-03],
       [-6.48333e-03, -9.27230e-03, -6.73866e-03, ..., -9.31693e-04,
         9.34093e-03,  9.53047e-01]])

In [13]:
from limix.qtl import scan
from numpy import concatenate, newaxis
from numpy import eye
import random

A = np.matrix('0 1; 1 0')
A0 = np.ones((len(traits), 1)) 
A1 = np.eye(len(traits))

In [101]:
!pip install numpy==1.16.2

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 5.1 MB 17.4 MB/s eta 0:00:01
  ERROR: Command errored out with exit status 1:
   command: /software/f2021/software/anaconda3/2021.11/bin/python -u -c 'import io, os, sys, setuptools, tokenize; sys.argv[0] = '"'"'/tmp/pip-install-42pfms43/numpy_67c7d0f28f8c4bba87304494709bb1ae/setup.py'"'"'; __file__='"'"'/tmp/pip-install-42pfms43/numpy_67c7d0f28f8c4bba87304494709bb1ae/setup.py'"'"';f = getattr(tokenize, '"'"'open'"'"', open)(__file__) if os.path.exists(__file__) else io.StringIO('"'"'from setuptools import setup; setup()'"'"');code = f.read().replace('"'"'\r\n'"'"', '"'"'\n'"'"');f.close();exec(compile(code, __file__, '"'"'exec'"'"'))' bdist_wheel -d /tmp/pip-wheel-ys_myh3j
       cwd: /tmp/pip-install-42pfms43/numpy_67c7d0f28f8c4bba87304494709bb1ae/
  Complete output (129 lines):
  Running from numpy source directory.
  /tmp/pip-install-42pfms43/numpy_67c7d0f28f8c4bba8

In [58]:
# Quick data check
print("\nData check:")
print(f"G shape: {G.shape}")
print(f"K shape: {K_sub.shape}")
print(f"Phenotype shape: {pheno.shape}")

# Check for any obvious issues
print("\nData validation:")
print(f"G contains NaN: {np.isnan(G).any()}")
print(f"K contains NaN: {np.isnan(K_sub).any()}")
print(f"Phenotype contains NaN: {np.isnan(pheno).any()}")


Data check:
G shape: (266, 943557)
K shape: (266, 266)
Phenotype shape: (266, 2)

Data validation:
G contains NaN: False
K contains NaN: False
Phenotype contains NaN: 1    False
2    False
dtype: bool


In [59]:
import limix
import numpy as np
import pandas as pd

def run_gwas_simple(G, K, phenotype, output_prefix="gwas_results"):
    """
    Run GWAS using Limix (using correct interface)

    Parameters:
    -----------
    G : numpy array
        Genotype matrix (samples × SNPs)
    K : numpy array
        Kinship matrix (samples × samples)
    phenotype : numpy array or pandas Series
        Phenotype values
    output_prefix : str
        Prefix for output files

    Returns:
    --------
    DataFrame with GWAS results
    """
    # Convert inputs to numpy arrays and ensure correct dimensions
    G = np.array(G, dtype=float)
    K = np.array(K, dtype=float)
    y = np.array(phenotype, dtype=float).reshape(-1, 1)  # Make it 2D column vector

    # Create fixed effect design matrix (intercept)
    X = np.ones((G.shape[0], 1))

    print("Data shapes:")
    print(f"G: {G.shape}")
    print(f"K: {K.shape}")
    print(f"y: {y.shape}")
    print(f"X: {X.shape}")

    try:
        # Run GWAS using correct Limix interface
        print("Running GWAS...")
        gwas = limix.qtl.scan(G=G, Y=y, K=K, M=X, verbose=True)

        # Convert results to DataFrame
        results = pd.DataFrame({
            'SNP': range(G.shape[1]),
            'Pvalue': gwas.stats.pv[0],  # First column of p-values
            'Beta': gwas.stats.beta[0],   # Effect sizes
            'SE': gwas.stats.se[0]        # Standard errors
        })

        # Save results
        results.to_csv(f"{output_prefix}.csv", index=False)
        print(f"Results saved to {output_prefix}.csv")

        # Print top associations
        print("\nTop 10 associations:")
        top_results = results.nsmallest(10, 'Pvalue')
        print(top_results)

        return results

    except Exception as e:
        print(f"Error during GWAS: {str(e)}")
        print("Debugging information:")
        print(f"G contains NaN: {np.isnan(G).any()}")
        print(f"K contains NaN: {np.isnan(K).any()}")
        print(f"y contains NaN: {np.isnan(y).any()}")
        print(f"G range: {np.min(G)} to {np.max(G)}")
        print(f"K range: {np.min(K)} to {np.max(K)}")
        print(f"y range: {np.min(y)} to {np.max(y)}")
        raise

# First, check your phenotype data
print("Checking phenotype data:")
print(pheno.head())
print("\nPhenotype shape:", pheno.shape)

# If phenotype is a DataFrame with multiple columns, select one column
if isinstance(pheno, pd.DataFrame) and pheno.shape[1] > 1:
    phenotype_values = pheno.iloc[:, 0]  # Use first column
else:
    phenotype_values = pheno

# Run GWAS
results = run_gwas_simple(
    G=G,
    K=K_sub,
    phenotype=pheno,
    output_prefix="ddm1_gwas_results"
)

Checking phenotype data:
              1         2
0                        
10010  0.024980 -0.324621
1002  -0.049857 -0.024401
108   -0.033401  1.064215
14319 -0.040161  0.492114
15560 -0.016326  0.265559

Phenotype shape: (266, 2)
Data shapes:
G: (266, 943557)
K: (266, 266)
y: (532, 1)
X: (266, 1)
Running GWAS...


================================= QTL analysis starts ==================================

Normalising input... 


failed (0.02 seconds).

========================== QTL analysis fails in 0.03 seconds ==========================

Error during GWAS: expected string or bytes-like object
Debugging information:
G contains NaN: False
K contains NaN: False
y contains NaN: False
G range: -2.0 to 2.0
K range: -0.0679109 to 5.52756
y range: -0.772384841704681 to 1.60247241607548


TypeError: expected string or bytes-like object

In [61]:
limix

<module 'limix' from '/users/elizaveta.grigoreva/.local/lib/python3.9/site-packages/limix/__init__.py'>

In [65]:
import limix
import numpy as np
import pandas as pd

def run_gwas_simple(G, K, phenotype, output_prefix="gwas_results"):
    """
    Run GWAS using Limix with proper data type handling
    """
    # Convert inputs to numpy arrays and ensure correct dimensions
    G = np.array(G, dtype=float)
    K = np.array(K, dtype=float)
    
    # If phenotype is a DataFrame, convert to numpy array
    if isinstance(phenotype, pd.DataFrame):
        y = phenotype.values
    else:
        y = np.array(phenotype)
    
    # Ensure y is 2D
    if len(y.shape) == 1:
        y = y.reshape(-1, 1)
    
    # Create fixed effect design matrix (intercept)
    X = np.ones((G.shape[0], 1))
    
    print("Data shapes:")
    print(f"G: {G.shape}")
    print(f"K: {K.shape}")
    print(f"y: {y.shape}")
    print(f"X: {X.shape}")
    
    try:
        # Run GWAS using Limix QTL scan
        print("Running GWAS...")
        gwas = limix.qtl.scan(
            G=G,                  # SNPs
            Y=y.astype(float),    # Phenotype
            K=K,                  # Kinship matrix
            M=X,                  # Covariates
            verbose=True
        )
        
        # Convert results to DataFrame
        results = pd.DataFrame({
            'SNP': range(G.shape[1]),
            'Pvalue': gwas.stats.pv[0],
            'Beta': gwas.stats.beta[0],
            'SE': gwas.stats.se[0]
        })
        
        # Save results
        results.to_csv(f"{output_prefix}.csv", index=False)
        print(f"Results saved to {output_prefix}.csv")
        
        return results
    
    except Exception as e:
        print(f"Error during GWAS: {str(e)}")
        print("\nDebugging information:")
        print(f"G contains NaN: {np.isnan(G).any()}")
        print(f"K contains NaN: {np.isnan(K).any()}")
        print(f"y contains NaN: {np.isnan(y).any()}")
        print(f"G range: {np.min(G)} to {np.max(G)}")
        print(f"K range: {np.min(K)} to {np.max(K)}")
        print(f"y range: {np.min(y)} to {np.max(y)}")
        print("\nData types:")
        print(f"G dtype: {G.dtype}")
        print(f"K dtype: {K.dtype}")
        print(f"y dtype: {y.dtype}")
        raise

# Select first phenotype if multiple exist
if isinstance(pheno, pd.DataFrame) and pheno.shape[1] > 1:
    print(f"Multiple phenotypes detected. Using first phenotype: {pheno.columns[0]}")
    phenotype_values = pheno.iloc[:, 0]
else:
    phenotype_values = pheno

# Run GWAS
try:
    results = run_gwas_simple(
        G=G,
        K=K_sub,
        phenotype=phenotype_values,
        output_prefix="ddm1_gwas_results"
    )
    print("\nGWAS completed successfully!")
except Exception as e:
    print(f"\nGWAS failed with error: {str(e)}")

Multiple phenotypes detected. Using first phenotype: 1
Data shapes:
G: (266, 943557)
K: (266, 266)
y: (266, 1)
X: (266, 1)
Running GWAS...


================================= QTL analysis starts ==================================

Normalising input... 


failed (0.01 seconds).

========================== QTL analysis fails in 0.02 seconds ==========================

Error during GWAS: expected string or bytes-like object

Debugging information:
G contains NaN: False
K contains NaN: False
y contains NaN: False
G range: -2.0 to 2.0
K range: -0.0679109 to 5.52756
y range: -0.103047150623329 to 0.533678753218824

Data types:
G dtype: float64
K dtype: float64
y dtype: float64

GWAS failed with error: expected string or bytes-like object


In [64]:
!pip install --upgrade limix

Defaulting to user installation because normal site-packages is not writeable


In [1]:
from limix.qtl import scan
r = scan(G, pheno, K = K_sub, lik = 'normal', A = A, A0 = A0, A1 = A1, verbose = True)

NameError: name 'G' is not defined

In [43]:
pheno

,1,2
0,,
b'10010',0.024980,-0.324621
b'1002',-0.049857,-0.024401
b'108',-0.033401,1.064215
b'14319',-0.040161,0.492114
b'15560',-0.016326,0.265559
...,...,...
b'9927',0.015342,-0.406540
b'9933',0.072331,0.145948
b'9938',-0.054522,0.056299


In [56]:
import limix
import numpy as np
import pandas as pd
import h5py


# 5. Run GWAS for first phenotype
y = pheno.iloc[:, 0].values.reshape(-1, 1)
X = np.ones((len(pheno), 1))  # Intercept

# Use qtl_test_lmm instead of scan
from limix.qtl import qtl_test_lmm
results = qtl_test_lmm(
    snps=G,
    pheno=pheno,
    K=K_sub,
    covs=X,
    test="lrt"
)



ImportError: cannot import name 'qtl_test_lmm' from 'limix.qtl' (/users/elizaveta.grigoreva/.local/lib/python3.9/site-packages/limix/qtl/__init__.py)

In [47]:
import limix
print(f"Limix version: {limix.__version__}")

Limix version: 3.0.4


In [55]:
pheno

,1,2
0,,
10010,0.024980,-0.324621
1002,-0.049857,-0.024401
108,-0.033401,1.064215
14319,-0.040161,0.492114
15560,-0.016326,0.265559
...,...,...
9927,0.015342,-0.406540
9933,0.072331,0.145948
9938,-0.054522,0.056299
